In [3]:
import pandas as pd
import numpy as np
from rdkit.Chem import MolFromSmiles 
import os
import pickle as pkl
from pathlib import Path
path_to_data = f'../../../data/neims/gecko_EIMS_spectra'
assert Path(path_to_data).exists()
ENABLE_FILE_EXPORTS = True 

In [4]:
from sys import version
py_ver = version.replace(' ', '').split('|')[0].replace('.', '_')
py_ver

'3_9_22'

In [5]:
smiles = pd.read_csv(f'{path_to_data}/processed/SMILES_gecko_corr_nitrate.csv')
print(f'NUM MOLS (SMILES): {len(smiles)}')
mols = []
for smi in smiles.values.flatten():
    try:
        mols.append(MolFromSmiles(smi))
    except: 
        print('Issue with generating mol from smiles')

NUM MOLS (SMILES): 166434


In [6]:
def file_exists(fpath: str):
    if os.path.exists(fpath + '/annotated.sdf'):
        #print("The file exists.")
        return True
    else:
        #print("The file does not exist.")
        return False
def rdkit_3d_exists(lines: str):
    found_rdkit = False
    found_3d = False
    for line in lines:
        if 'rdkit' in line:
            found_rdkit = True
        if '3d' in line:
            found_3d = True
    return found_rdkit and found_3d
def spectrum_exists(lines: str):
    spec_num_peaks = 0
    found_spec = False
    for idx, line in enumerate(lines):
        if found_spec and line != '$$$$\n' and line != '\n':
            spec_num_peaks += 1
        if 'predicted spectrum' in line:
            found_spec = True  
    return found_spec, spec_num_peaks

In [7]:
nums = np.arange(2, 166436) 
exists_idx = []
not_exists_idx = []
folders = []
for idx, num in enumerate(nums):
    default = f'000000'
    folder = f''
    num_str = str(num)[::-1]
    for i, char in enumerate(default):
        try:
            folder += num_str[i]
        except:
            folder += '0'
    folder = folder[::-1]
    folders.append(folder)

for idx, folder in enumerate(folders):
    if file_exists(f'{path_to_data}/processed/smiles_split/{folder}'):
        exists_idx.append(idx)
    else:
        not_exists_idx.append(idx)
print(len(exists_idx), 'exists')
print(len(not_exists_idx), 'not exists')

166434 exists
0 not exists


In [ ]:
df = pd.DataFrame()
df['SMILES'] = smiles
specs = []
for idx, folder in enumerate(folders):
    if idx in not_exists_idx:
        specs.append(None)
        continue
    filename = f'{path_to_data}/processed/smiles_split/{folder}/annotated.sdf'
    #print(filename)
    with open(filename, 'r') as file:
        lines = file.readlines()
        lines = [line.lower() for line in lines]
    #print(lines)
    #print(len(lines))
    #print(lines)
    spectrum_found, spec_num_peaks = spectrum_exists(lines)
    assert rdkit_3d_exists(lines), 'rdkit 3d must exist'
    assert spectrum_exists(lines), 'spectrum must exist'
    assert spec_num_peaks >= 5, 'spectrum must have more at least 5 peaks'
    spec_flag = False
    spec = [[],[]]
    for line in lines:
        if 'predicted spectrum' in line:
            spec_flag = True
            continue
        if spec_flag and line != '$$$$\n' and line != '\n':
            location, intensity = line.replace('\n', '').split()
            spec[0].append(location); spec[1].append(intensity) 
    specs.append(np.array(spec, dtype=np.uint16).T)
df['spec'] = specs
if ENABLE_FILE_EXPORTS:
    df.to_csv('df_neims_gecko.csv', index=False)
    df.to_pickle(f"df_neims_gecko_{py_ver}.pkl")

In [9]:
smiles = pd.read_csv(f'{path_to_data}/TMS/SMILES_gecko_TMS_corr_nitrate.csv')
print(f'NUM MOLS (SMILES): {len(smiles)}')
mols = []
for smi in smiles.values.flatten():
    try:
        mols.append(MolFromSmiles(smi))
    except: 
        print('Issue with generating mol from smiles')

NUM MOLS (SMILES): 166434


In [10]:
nums = np.arange(2, 166436) 
exists_idx = []
not_exists_idx = []
folders = []
for idx, num in enumerate(nums):
    default = f'000000'
    folder = f''
    num_str = str(num)[::-1]
    for i, char in enumerate(default):
        try:
            folder += num_str[i]
        except:
            folder += '0'
    folder = folder[::-1]
    folders.append(folder)

for idx, folder in enumerate(folders):
    if file_exists(f'{path_to_data}/TMS/smiles_split/{folder}'):
        exists_idx.append(idx)
    else:
        not_exists_idx.append(idx)
print(len(exists_idx), 'exists')
print(len(not_exists_idx), 'not exists')

166434 exists
0 not exists


In [ ]:
tms_df = pd.DataFrame()
tms_df['SMILES'] = smiles
specs = []
for idx in exists_idx:
    filename = f'{path_to_data}/TMS/smiles_split/{folders[idx]}/annotated.sdf'
    #print(filename)
    with open(filename, 'r') as file:
        lines = file.readlines()
        lines = [line.lower() for line in lines]
    #print(lines)
    #print(len(lines))
    #print(lines)
    spectrum_found, spec_num_peaks = spectrum_exists(lines)
    assert rdkit_3d_exists(lines), 'rdkit 3d must exist'
    assert spectrum_exists(lines), 'spectrum must exist'
    assert spec_num_peaks >= 5, 'spectrum must have at least 5 peaks'
    spec_flag = False
    spec = [[],[]]
    for line in lines:
        if 'predicted spectrum' in line:
            spec_flag = True
            continue
        if spec_flag and line != '$$$$\n' and line != '\n':
            location, intensity = line.replace('\n', '').split()
            spec[0].append(location); spec[1].append(intensity) 
    specs.append(np.array(spec, dtype=np.uint16).T)
tms_df['spec'] = specs

In [12]:
if ENABLE_FILE_EXPORTS:
    tms_df.to_csv('df_neims_gecko_TMS.csv', index=False)
    tms_df.to_pickle(f"df_neims_gecko_TMS_{py_ver}.pkl")

In [13]:
df

,SMILES,spec
0,C(=O),"[[14, 202], [15, 313], [16, 165], [18, 16], [1..."
1,C(=O)([N+](=O)[O-]),"[[14, 72], [16, 151], [17, 70], [18, 108], [19..."
2,C(=O)([N+](=O)[O-])C(=O),"[[14, 153], [15, 174], [16, 125], [17, 36], [1..."
3,C(=O)([N+](=O)[O-])C(=O)(O),"[[14, 120], [15, 236], [16, 149], [17, 78], [1..."
4,C(=O)([N+](=O)[O-])C(=O)(OO),"[[14, 136], [15, 268], [16, 152], [17, 62], [1..."
...,...,...
166429,CC1(O)C(O)(C(=O)(OO[N+](=O)[O-]))OOC1C(O)C(=O)...,"[[14, 70], [15, 145], [16, 82], [17, 31], [18,..."
166430,CC1(O)C(O)(C(=O)(OO[N+](=O)[O-]))OOC1C(O)C(=O),"[[14, 53], [15, 108], [16, 44], [17, 39], [18,..."
166431,CC1(O)C(O)(C(=O)(OO[N+](=O)[O-]))OOC1C(=O)C(=O),"[[14, 54], [15, 123], [16, 54], [17, 45], [18,..."
166432,CC1(O)C(O)(C(=O)(OO[N+](=O)[O-]))OOC1C(=O)(OO[...,"[[14, 60], [15, 69], [16, 69], [26, 44], [27, ..."
